# yolo_lpr_cpp — GPU 経路の確認 (CUDA)

**問い: 同じソース・同じ ONNX が、CPU と GPU で同じ数値を出すか。**

エンジンの device seam は `pure/backend.hpp` だけ（`bk::gemm_hosted` / `gemm_nt_hosted` /
`gemm_tn_hosted`）。conv2d の forward も backward の 2 つの積も、線形層も、全部そこを通る。
`-DUSE_CUDA` を付けると同じ関数が device メモリにステージして CUDA カーネルを呼ぶだけなので、
アーキテクチャも学習コードも **1 行も変わらない**。

だから確認はこうなる: `pure/gpu_check.cpp` を **CPU と GPU の両方でビルドして、出力を diff する**。
中身は検出器の forward（6 本の head テンソルのチェックサム）、学習 1 step の loss と勾配の L2、
4隅回帰の forward。

開発機（Windows）には CUDA ツールキットはあるが NVIDIA GPU が無いので、そこで確認できるのは
「nvcc でビルドが通る」ところまで。**実機での一致はこのノートで見る。**

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1

In [ ]:
!git clone -q https://github.com/yomei-o/yolo_lpr_cpp.git
%cd yolo_lpr_cpp

In [ ]:
# 検査用の合成フレームを数枚（CPU でも GPU でも同じ入力にするため、seed 固定で生成する）
!g++ -std=c++20 -O2 -Ipure -Ipure/third_party pure/jlpr.cpp -o jlpr
!./jlpr gen-det --out data/det_smoke --count 8 --imgsz 320 --seed 5 --quiet
!ls data/det_smoke/images | head -3

In [ ]:
# 同じソースを 2 通りにビルドする（CPU 側は Eigen 無し = 素の GEMM）
!g++ -std=c++20 -O2 -Ipure -Ipure/third_party pure/gpu_check.cpp -o gpu_check_cpu
!nvcc -x cu -std=c++17 -O2 --extended-lambda -DUSE_CUDA -arch=sm_75 \
      -Ipure -Ipure/third_party pure/gpu_check.cpp -o gpu_check_gpu
!ls -la gpu_check_cpu gpu_check_gpu

In [ ]:
!./gpu_check_cpu --data data/det_smoke | tee cpu.txt
print()
!./gpu_check_gpu --data data/det_smoke | tee gpu.txt

In [ ]:
# 数値の突き合わせ: backend 行以外は完全一致するはず（GEMM の順序差だけなので、
# 一致しない場合は下の相対差を見る。1e-4 を超えるなら実装の問題）
import re
def parse(p):
    out = {}
    for line in open(p):
        for k, v in re.findall(r"([a-z0-9 ]+?)\s+(-?\d+\.\d+)", line.lower()):
            out.setdefault(k.strip(), []).append(float(v))
    return out
c, g = parse("cpu.txt"), parse("gpu.txt")
worst = 0.0
for k in c:
    if k not in g:
        continue
    for a, b in zip(c[k], g[k]):
        rel = abs(a - b) / max(1e-9, abs(a))
        worst = max(worst, rel)
        if rel > 1e-4:
            print("DIFFERS %-28s cpu %.6f  gpu %.6f  rel %.2e" % (k, a, b, rel))
print("worst relative difference: %.3e" % worst)
print("PASS" if worst < 1e-4 else "FAIL")

## 学習も GPU で回すには

`pure/jlpr.cpp` も同じ `nvcc -x cu -DUSE_CUDA` で通る（同じ seam しか使っていないので）。
ただし現在の CUDA 経路は **GEMM ごとに host↔device をステージする**ので、小さい層が多い
yolov8n では転送が支配的になる。速度が要るなら `-DUSE_CUBLAS -lcublas` を足すか、
姉妹リポ（lpr_cpp / facenet_cpp）と同じように活性値を device に置きっぱなしにする device 版の
forward/backward を書くことになる。**このノートの目的は数値の一致**で、速度ではない。

In [ ]:
# 参考: CLI 全体を CUDA でビルドして 1 step だけ回す（時間はかかる。数値が同じことの確認用）
!nvcc -x cu -std=c++17 -O2 --extended-lambda -DUSE_CUDA -arch=sm_75 \
      -Ipure -Ipure/third_party pure/jlpr.cpp -o jlpr_gpu
!./jlpr_gpu train --model det --data data/det_smoke --limit 2 --batch 2 --steps 2 --no-aug --no-ema
!./jlpr    train --model det --data data/det_smoke --limit 2 --batch 2 --steps 2 --no-aug --no-ema